[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/penrose/penrose.ipynb)

# Penrose Tilings

A Penrose tiling covers the plane with two kinds of rhombus, a thick and a thin one, in a pattern that never repeats yet has five-fold symmetry everywhere. It is the shadow of an ordinary lattice in more dimensions: the cubes of the integer lattice of five-dimensional space, cut by a plane tilted so that it never passes through a second lattice point. Each square face of the lattice that the plane passes close enough to becomes a tile, laid flat onto the plane. Shifting the plane without tilting it changes which faces are close enough, and the tiling rearranges itself by local flips, never sliding as a whole.

The lattice's points and axes are antivectors, intrinsic like all geometry. Turning the five axes into one another in a cycle is a rotation, the product of four reflections. It turns one plane by a fifth of a turn, another by two fifths, and leaves the diagonal fixed; averaged over its five powers with the right weights, it projects onto the first plane, `along(Antivector)`, and the cut is the join of two axes laid flat, `along(first) & along(second)`. A face is a tile where the cut passes through the cube it is dual to, and the planes that read the cube's three axes, blind to the cut, tell whether it does: Cramer's rule with joins alone, `(second & third & cut) / (first & second & third & cut)`, with no metric. How often a face is a tile is the size of its dual cube's shadow square to the cut, which equals the size of its tile: thick tiles outnumber thin ones by the golden ratio.

In the notation of de Bruijn's pentagrid, a tiling vertex is the integer point $k_j = \lceil \mathbf x \cdot \mathbf e_j + \gamma_j \rceil$ of the five grids of lines $\mathbf x \cdot \mathbf e_j + \gamma_j \in \mathbb Z$, with $\mathbf e_j = (\cos\frac{2\pi j}{5}, \sin\frac{2\pi j}{5})$ and $\sum_j \gamma_j = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import itertools

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure, show_animation
from examples.math.penrose import render

np.set_printoptions(precision=3, suppress=True)

ga = Algebra("a+b+c+d+e+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Planes, read by pairing with points: the mirrors, and the planes that read a point's offsets.
Plane = ga.gatype.vector()
# Points and directions are antivectors, intrinsic; the join of two is a plane through them.
Antivector = ga.gatype.antivector()
Join = ga.gatype.antibivector()
Rotor = ga.gatype.rotor()
Projection = ga.gatype((Antivector, Antivector))          # Antivector <- Antivector: onto one of the turn's planes

mirrors = mv.basis()                                      # [5] Plane: across each axis
axes = mirrors.dual()                                     # [5] Antivector
# The faces of the lattice: each pair of axes, and the three other axes of each.
faces_axes = list(itertools.combinations(range(5), 2))
sides_index = np.array(faces_axes)                        # [faces, 2]
others_index = np.array([[k for k in range(5) if k not in face] for face in faces_axes])   # [faces, 3]

## 1. The turn and the cut

Reflecting in the mirror between two neighbouring axes swaps them, and four such swaps in a row carry each axis to the next: a rotation of order five. As a map on directions, `turn() >> Antivector`, it can be applied again and again, its fifth power being the identity. The cosines of the multiples of a fifth of a turn are orthogonal over five steps, so the average of the five powers, each weighted by the cosine of that many fifths of a turn, keeps the plane that turns at the matching rate and cancels the others. Weighted by one fifth of a turn, it is the projection onto the cut, `along`; by two fifths, onto the plane square to it and to the diagonal, `across`. The cut itself is the join of two axes laid flat.

Laid flat onto the cut, the five axes are equally long and a fifth of a turn apart: their dot products are their common squared length, two fifths, times the cosines of one and two fifths of a turn.

In [ ]:
def turn() -> Rotor:
    """The rotation turning each axis into the next and the last into the first: four reflections."""
    # The mirror between two neighbouring axes swaps them.
    normals = (mirrors[:-1] - mirrors[1:]) / np.sqrt(2)                       # [4] Plane
    return normals[0] * normals[1] * normals[2] * normals[3]                  # [] Rotor


def powers(shift: Projection, n_powers: int):
    """The map applied once, twice, and so on."""
    power = shift                                                             # [] Antivector <- Antivector
    for _ in range(n_powers):
        yield power
        power = shift(power)


def projection(harmonic: int) -> Projection:
    """The projection onto the plane the turn turns by `harmonic` fifths of a turn: its five powers as
    maps on directions, the fifth being the identity, each weighted by the cosine of that turn times the power."""
    maps = stack(list(powers(turn() >> Antivector, 5)))                       # [5] Antivector <- Antivector
    weights = 2 / 5 * np.cos(2 * np.pi * harmonic * np.arange(1, 6) / 5)
    return (maps * weights).sum(axis=0)                                       # [] Antivector <- Antivector


along, across = projection(1), projection(2)                                  # [] Antivector <- Antivector each
cut = along(axes[0]) & along(axes[1])                                         # [] Join
print("the axes laid flat, against each other:\n", along(axes)[:, None].dot(along(axes)).to_array())

## 2. Tiles

A square face of the lattice becomes a tile where the cut passes through its dual cube: the unit cube through the face's centre along the three other axes. Whether it does is a question about the cut's offset from the face's centre along those three axes, ignoring any offset within the cut itself. The reciprocal frame of the three axes and the cut answers it: planes, each pairing to one with its axis and to nothing with the other two and the cut, built by Cramer's rule from joins alone, so each reading is a pairing, `frame & offset`. The face is a tile where all three readings are within half a unit.

The cut passes through `shift`, a point whose coordinates sum to zero, and every face of every lattice point within the radius is tested at once. Tiles are coloured by their area laid flat: the thick ones amber, the thin ones blue.

In [ ]:
# The lattice points within this radius of the origin along every axis, and the point the cut passes
# through, its coordinates summing to zero. Half the width of the view, in units of the tiles' edges.
lattice_radius = 3
shift = (axes * np.array([0.12, 0.29, -0.11, 0.33, -0.63])).sum()             # [] Antivector
extent = 2.6

In [ ]:
def reciprocal(cut: Join) -> Plane:
    """Per face, the reciprocal frame of its three other axes and the cut `[faces, 3]`: the planes whose
    pairings with a point read how far along each of the three axes it is, blind to the cut."""
    others = axes[others_index]                                                # [faces, 3] Antivector
    # Cramer's rule with joins: per axis, the other two joined with the cut, a plane that pairs to nothing
    # with them and with the cut, over the volume all three span with the cut.
    cofactors = others[:, [1, 2, 0]] & others[:, [2, 0, 1]] & cut             # [faces, 3] Plane
    volume = others[:, 0] & others[:, 1] & others[:, 2] & cut                 # [faces] Scalar
    return cofactors / volume[:, None]                                        # [faces, 3] Plane


def margins(shift: Antivector, centres: Antivector, frames: Plane) -> Scalar:
    """How far inside the dual cube of each face the cut passes, along each of its three other axes
    `[..., faces, 3]`, for the cut through `shift`: the face is a tile where all three are positive."""
    offsets = frames & (shift - centres)[..., None]                           # [..., faces, 3] Scalar
    return 1 / 4 - offsets.squared()                                          # [..., faces, 3] Scalar


def faces(along: Projection) -> tuple[Antivector, Antivector]:
    """Every square face of the lattice within the radius, by its lattice point and its pair of axes: its
    centre `[points, faces]`, and its corners laid flat onto the cut `[points, faces, 4]`."""
    sides = axes[sides_index]                                                 # [faces, 2] Antivector
    coordinates = np.array(list(itertools.product(range(-lattice_radius, lattice_radius + 1), repeat=5)))
    lattice = (axes * coordinates).sum(axis=-1)                               # [points] Antivector
    centres = lattice[:, None] + sides.sum(axis=-1) / 2                       # [points, faces] Antivector
    # The corners from the centre, half a unit back and forth along both axes, around the face.
    first, second = np.array([-1, 1, 1, -1]) / 2, np.array([-1, -1, 1, 1]) / 2
    corners = sides[:, None, 0] * first + sides[:, None, 1] * second          # [faces, 4] Antivector
    return centres, along(centres[..., None] + corners)


def shadows(along: Projection) -> tuple[Scalar, Scalar]:
    """Per face `[faces]`: its area laid flat onto the cut, and the volume of the shadow its dual cube
    casts square to the cut, how often the face is a tile among faces of every kind."""
    sides, others = axes[sides_index], axes[others_index]                     # [faces, 2], [faces, 3] Antivector
    # The tile is the join of the face's two sides laid flat; the shadow the join of its three other axes
    # with their parts in the cut taken away.
    square = Antivector - along                                               # [] Antivector <- Antivector
    tiles = along(sides[:, 0]) & along(sides[:, 1])                           # [faces] Join
    shadows = square(others[:, 0]) & square(others[:, 1]) & square(others[:, 2])   # [faces] Antitrivector
    return tiles.norm(), shadows.norm()


frames = reciprocal(cut)                                                      # [faces, 3] Plane
centres, corners = faces(along)
areas, volumes = shadows(along)                                               # [faces] Scalar each
# The first axis laid flat, and that turned a quarter turn in the cut by the commutator of two directions
# in it; the planes across them read the tiling.
right = along(axes[0]) / along(axes[0]).norm()                                # [] Antivector
turning = right.commutator(along(axes[1]))                                    # [] Bivector
up = (turning / turning.scalar_norm_squared().square_root() * (-np.pi / 4)).exp() >> right   # [] Antivector
reading = stack([right, up]).dual()                                           # [2] Plane
selected = margins(shift, centres, frames)                                    # [points, faces, 3] Scalar
save_figure(render.draw_tiling(reading, corners, areas, selected, extent), "penrose_tiling");

Over all the lattice points, the cut's offset from a face's centre, read square to the cut, is spread evenly over the three dimensions square to it, and the face is a tile where that offset falls inside its dual cube's shadow there, so it is a tile as often as that shadow is large: the join of the cube's three axes with their parts in the cut taken away, as the tile is the join of the face's two sides laid flat. A face and its dual cube together span the whole space, as do the cut and the space square to it, so the two shadows are equally large: each kind of face is a tile as often as its tile is large. The thick tile is the larger by the golden ratio, and so it is the more common by the golden ratio, which the patch above follows to within its edge.

In [ ]:
golden = (1 + np.sqrt(5)) / 2
thick = areas.to_array() > areas.to_array().mean()                            # [faces]
tiles = np.all(selected.to_array() > 0, axis=-1)                              # [points, faces]
print("tile areas:          ", areas.to_array())
print("dual cube shadows:   ", volumes.to_array())
print("thick against thin:  ", areas.to_array()[thick][0] / areas.to_array()[~thick][0], "golden ratio:", golden)
print("tiles in the patch:  ", tiles[:, thick].sum(), "thick,", tiles[:, ~thick].sum(), "thin")

## 3. Phason flips

Shifting the cut within itself only slides the tiling, since the reciprocal frames do not see the cut. Shifting it square to itself, in the plane `across` projects onto, changes which faces are close enough: a few tiles at a time flip, three rhombi around a vertex trading places inside the hexagon they fill, while the rest of the tiling stays put. Around a closed loop of shifts, every flip is undone and the tiling returns to where it started. This cell takes about four seconds, most of it drawing the frames.

In [ ]:
# How far the cut is shifted around its loop, the frames around it, and the duration of a frame.
phason = 0.35
n_frames = 90
duration_ms = 60

In [ ]:
def phasons(across: Projection) -> Antivector:
    """The points the cut passes through around its loop `[frames]`: the starting point shifted by a
    turn in the plane square to the cut and to the diagonal."""
    start = across(axes[0])                                                   # [] Antivector
    loop = start.commutator(across(axes[1]))                                  # [] Bivector
    angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
    turned = (loop / loop.scalar_norm_squared().square_root() * (angles / 2)).exp() >> start   # [frames] Antivector
    return shift + phason * (turned - start) / start.norm()                   # [frames] Antivector


def flips(shifts: Antivector, centres: Antivector, frames: Plane):
    """Per frame, the margins of every face within the lattice's radius `[points, faces, 3]`."""
    for shifted in shifts:
        yield margins(shifted, centres, frames)


show_animation(render.animate_flips(reading, corners, areas, flips(phasons(across), centres, frames), extent),
               "penrose_phasons", duration_ms)

In [ ]:
# checks
# The turn is of order five, and the reciprocal frame of each face reads its three other axes as a unit
# frame and the cut as nothing.
fifth = list(powers(turn() >> Antivector, 5))[-1]                             # [] Antivector <- Antivector
np.testing.assert_allclose((fifth(axes) - axes).norm().to_array(), 0, atol=1e-12)
others = axes[others_index]                                                   # [faces, 3] Antivector
np.testing.assert_allclose((frames[:, None] & others[..., None]).to_array(),
                           np.broadcast_to(np.eye(3), (len(faces_axes), 3, 3)), atol=1e-12)
np.testing.assert_allclose((frames & along(axes)[:, None, None]).to_array(), 0, atol=1e-12)
# Each dual cube's shadow is as large as its face laid flat, thick against thin in the golden ratio.
np.testing.assert_allclose(volumes.to_array(), areas.to_array(), atol=1e-12)
np.testing.assert_allclose(areas.to_array()[thick] / areas.to_array()[~thick][:, None], golden, rtol=1e-12)